In [27]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import os
import time

In [15]:
FILE_PATH_LINKS = "proceduces_links.csv"
BASE_URL = "https://dichvucong.gov.vn/p/home/dvc-danh-sach-thu-tuc-theo-nhom-su-kien.html?objectId=1&groupEventId=753#mainTitle"
DRIVER_PATH = r"D:\chromedriver\chromedriver-win64\chromedriver.exe"
DOWNLOAD_DIR = os.path.abspath("downloads")
data = []

In [17]:
def make_driver(driver_path: str):
    options = webdriver.ChromeOptions()
    # options.add_argument("--headless")         
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-images")    
    options.add_argument("--blink-settings=imagesEnabled=false")

    # Cấu hình download
    prefs = {
        "download.default_directory": DOWNLOAD_DIR,
        "download.prompt_for_download": False,
        "plugins.always_open_pdf_externally": True 
    }
    options.add_experimental_option("prefs", prefs)
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)

    s = Service(driver_path)
    return webdriver.Chrome(service=s, options=options)

In [ ]:
# def get_links(base_url: str, file_name: str):
    
#     driver = make_driver(DRIVER_PATH)
#     wait = WebDriverWait(driver, 10)
#     total_links = 0
#     links = []

#     driver.get(base_url)
#     ul = wait.until(EC.presence_of_element_located((By.ID, "ulTemplate")))
#     items = ul.find_elements(By.TAG_NAME, "li")

#     for item in items:
#         li_tag = item.find_element(By.TAG_NAME, "a")
#         link = li_tag.get_attribute("href")
#         links.append(link)

#     df = pd.DataFrame(links)
#     df.to_csv(file_name, index=False, header=False, mode="a")   
#     total_links += len(links)
#     print(f"Đã xong, lấy được {len(links)} links")

#     driver.quit()

In [ ]:
def get_links(base_url: str, file_name: str):
    driver = make_driver(DRIVER_PATH)
    wait = WebDriverWait(driver, 15)

    all_links = []
    seen = set()

    def extract_links_current_page():
        ul = wait.until(EC.presence_of_element_located((By.ID, "ulTemplate")))
        items = ul.find_elements(By.TAG_NAME, "li")

        page_links = []
        for item in items:
            try:
                a_tag = item.find_element(By.TAG_NAME, "a")
                link = a_tag.get_attribute("href")
                if link and link not in seen:
                    seen.add(link)
                    page_links.append(link)
            except Exception:
                continue

        return page_links

    try:
        driver.get(base_url)

        page_1_links = extract_links_current_page()
        all_links.extend(page_1_links)
        print(f"Trang 1: lấy được {len(page_1_links)} links")

        for page_num in [2, 3]:
            try:
                old_first_link = wait.until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "#ulTemplate li a"))
                )

                page_btn = wait.until(
                    EC.element_to_be_clickable(
                        (
                            By.XPATH,
                            f"//a[normalize-space()='{page_num}'] | "
                            f"//button[normalize-space()='{page_num}'] | "
                            f"//*[self::a or self::button or self::span][normalize-space()='{page_num}']"
                        )
                    )
                )

                driver.execute_script("arguments[0].click();", page_btn)

                wait.until(EC.staleness_of(old_first_link))

                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#ulTemplate li a")))

                page_links = extract_links_current_page()
                all_links.extend(page_links)
                print(f"Trang {page_num}: lấy được {len(page_links)} links")

            except TimeoutException:
                print(f"Không tìm được hoặc không click được trang {page_num}")
                break

        df = pd.DataFrame(all_links)
        df.to_csv(file_name, index=False, header=False)

        print(f"Đã xong, tổng số links lấy được: {len(all_links)}")

    finally:
        driver.quit()

In [47]:
def wait_for_download(download_dir, filename, timeout=30):
    path = os.path.join(download_dir, filename)
    start = time.time()

    while time.time() - start < timeout:
        if os.path.exists(path):
            tmp_files = [f for f in os.listdir(download_dir) if f.endswith(('.crdownload', '.tmp'))]
            if not tmp_files:
                pdf_path = os.path.splitext(path)[0] + ".pdf"
                return pdf_path
        time.sleep(0.5)

    print(f"Timeout chờ download: {filename}")
    return None

In [30]:
def parse_table(table, driver=None, download_col="Mẫu đơn, tờ khai"):
    headers = []
    try:
        thead = table.find_element(By.TAG_NAME, "thead")
        if thead:
            headers = [th.get_attribute("textContent").strip() for th in thead.find_elements(By.TAG_NAME, "th")]
    except:
        return []

    rows = []
    try:
        tbody = table.find_element(By.TAG_NAME, "tbody")
        if tbody:
            for tr in tbody.find_elements(By.TAG_NAME, "tr"):
                tds = tr.find_elements(By.TAG_NAME, "td")
                cells = []

                for i, td in enumerate(tds):
                    col_name = headers[i] if i < len(headers) else ""

                    if col_name == download_col and driver:
                        spans = td.find_elements(By.TAG_NAME, "span")
                        file_paths = []

                        for span in spans:
                            file_name = span.get_attribute("textContent").strip()
                            if not file_name:
                                continue

                            try:
                                old_path = os.path.join(DOWNLOAD_DIR, file_name)
                                if os.path.exists(old_path):
                                    os.remove(old_path)

                                driver.execute_script("arguments[0].click();", span)

                                downloaded_path = wait_for_download(DOWNLOAD_DIR, file_name)
                                file_paths.append(downloaded_path or file_name)

                            except Exception as e:
                                print(f"Lỗi download {file_name}:", e)
                                file_paths.append(file_name)

                        cells.append(", ".join(file_paths))
                    else:
                        cells.append(td.get_attribute("textContent").strip())

                if cells and len(cells) == len(headers):
                    rows.append(dict(zip(headers, cells)))
    except:
        pass

    return rows

In [31]:
def parse_report_components(container, driver):
    result = {}
    col_sm9 = container.find_element(By.CSS_SELECTOR, ".col-sm-9")
    children = col_sm9.find_elements(By.XPATH, "./*")
    current_key = "Mặc định"

    for child in children:
        tag = child.tag_name
        class_attr = child.get_attribute("class") or ""

        if tag == "div" and "key" in class_attr:
            current_key = child.get_attribute("textContent").strip()
        elif tag == "table":
            rows = parse_table(child, driver=driver)  
            result[current_key] = rows

    return result

In [32]:
import json

# BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")

RAW_DATA_DIR = os.path.join("data", "raw")

os.makedirs(RAW_DATA_DIR, exist_ok=True)

def parse_listing(links: list[str]):
    driver = make_driver(DRIVER_PATH)

    for link in links:
        driver.get(link)

        a_tag = driver.find_element(By.CLASS_NAME, "url")
        driver.get(a_tag.get_attribute("href"))
        
        wait = WebDriverWait(driver, 10)

        attributes = wait.until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".info-row"))
        )
        procedure = {}
        for attribute in attributes:
            try:
                key = attribute.find_element(By.CSS_SELECTOR, ".col-sm-3.col-xs-12.key").get_attribute("textContent").strip()
                list_key_table = ["Cách thức thực hiện:", "Căn cứ pháp lý:"]
                value_element = attribute.find_element(By.CSS_SELECTOR, ".col-sm-9")
                value = value_element.get_attribute("textContent").strip()

                if key in list_key_table:
                    procedure[key] = parse_table(attribute)
                elif key == "Thành phần hồ sơ:":
                    procedure[key] = parse_report_components(attribute, driver)
                else:
                    procedure[key] = value

                if key == "Mã thủ tục:":
                    PROCEDUCES_PATH = f"{value}.json"

            except Exception as e:
                print("Lỗi tại row:", attribute.get_attribute("outerHTML"))
                print("Error:", e)
            
        
        with open(PROCEDUCES_PATH, "a", encoding="utf-8") as file:
            json.dump(procedure, file, indent=4, ensure_ascii=False)
    driver.quit()
    

In [45]:
links = ["https://dichvucong.gov.vn/p/home/dvc-chi-tiet-thu-tuc-hanh-chinh.html?ma_thu_tuc=1.004194"]
# get_links(BASE_URL, FILE_PATH_LINKS)
parse_listing(links)

File đã tồn tại, bỏ qua: d:\capstone-project\notebooks\downloads\1.MuCT01banhnhkmtheoThngts53.pdf
File đã tồn tại, bỏ qua: d:\capstone-project\notebooks\downloads\1.MuCT01banhnhkmtheoThngts53.pdf
File đã tồn tại, bỏ qua: d:\capstone-project\notebooks\downloads\1.MuCT01banhnhkmtheoThngts53.pdf


In [84]:
import fitz 

doc = fitz.open("1.MuCT01banhnhkmtheoThngts53.pdf")
fulltext = ""
for page in doc:
    text = page.get_text()
    fulltext += text

In [ ]:
print(fulltext)

In [ ]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import SystemMessage, HumanMessage
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_API_KEY,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

messages = [
    SystemMessage(content = f"""Bạn là chuyên gia về thủ tục hành chính Việt Nam, hãy đọc mẫu đơn từ và thông tin từ người dùng cung cấp sau đó điền mẫu đơn hoàn chỉnh\
        Nội dung mẫu đơn {fulltext}"""\
    ),
    HumanMessage(content = "Tôi tên Trần Đình Thắng, sinh ngày 27/07/2003, Nam")
]


ai_msg = llm.invoke(messages)
print(ai_msg.content)

In [ ]:
from dotenv import load_dotenv
import boto3
import os

load_dotenv() 

BUCKET = 'proceduces-forms'
REGION = 'ap-southeast-2'
KEY = '1.MuCT01banhnhkmtheoThngts53.pdf'

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_DEFAULT_REGION')
)

s3.upload_file('1.MuCT01banhnhkmtheoThngts53.pdf', 'proceduces-forms', KEY , ExtraArgs={
        'ContentType': 'application/pdf' 
    })
print("Upload thành công!")

url = f"https://{BUCKET}.s3.{REGION}.amazonaws.com/{KEY}"
print(url) 

## def gen_url_file(filepath: Path) -> str

Upload thành công!
https://proceduces-forms.s3.ap-southeast-2.amazonaws.com/1.MuCT01banhnhkmtheoThngts53.doc


In [9]:
import webbrowser

webbrowser.open(url)

True